# 3 · Predict wind maps over a large DEM

Produces one multi-band GeoTIFF per wind direction (bands named `<field>_<height>m`), ready to be stacked as inputs of the thermal model:
`u_east, v_north, w, tke, speed, speedup, dir_from, deflection, ti, convergence, recirculation, shadow, compression` at each height, plus `terrain_m`.
Fields scale linearly with the reference wind (`--uref` at `flow.Zref`, k with its square), so one run per direction (and z0) is enough.

In [ ]:
# --- Project location on Google Drive -------------------------------------
# Mounts Drive only if it is not mounted yet. NEVER run `rm -rf /content/drive`:
# if Drive is mounted it deletes your whole Google Drive.
import os, subprocess
if not os.path.ismount('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/windsurrogate'      # change if you like
PROJECT    = f'{DRIVE_ROOT}/idaho'                        # one folder per dataset (regions in config)
CODE_ZIP   = f'{DRIVE_ROOT}/windsurrogate_code.zip'       # upload the code zip here once
assert os.path.ismount('/content/drive'), 'Drive is not mounted'
assert os.path.exists(CODE_ZIP), f'missing {CODE_ZIP}: upload the code zip there first'
os.makedirs(PROJECT, exist_ok=True)

# --- Code ---------------------------------------------------------------------
if not os.path.exists('/content/windsurrogate'):
    subprocess.run(['unzip', '-q', '-o', CODE_ZIP, '-d', '/content'], check=True)
%cd /content/windsurrogate
!pip -q install rasterio pyproj pyyaml

Maps that already exist are skipped, so the cell can be re-run after a disconnect. Each map is written under a temporary name and only renamed when complete.
At 50 m, one direction takes about a minute per region on an A100 and roughly 0.5-1 GB per file.

In [ ]:
import os
CKPT = f'{PROJECT}/checkpoints/best.pt'
OUT  = f'{PROJECT}/predictions'
Z0 = 0.1                                 # or loop over a few roughness values
REGIONS = ['cervidae', 'lost_river']
DIRECTIONS = range(0, 360, 30)           # 12 directions; interpolate between neighbours downstream
for region in REGIONS:
    DEM = f'{PROJECT}/dem/{region}.tif'
    for theta in DIRECTIONS:
        out = f'{OUT}/{region}/wind_{theta:03d}.tif'
        if os.path.exists(out):
            continue
        !python -m windsurrogate predict --checkpoint {CKPT} --dem {DEM} --out {out} --theta {theta} --z0 {Z0}

In [ ]:
import rasterio, numpy as np, matplotlib.pyplot as plt
with rasterio.open(f'{OUT}/lost_river/wind_270.tif') as ds:
    names = list(ds.descriptions)
    get = lambda n: ds.read(names.index(n) + 1)
    fig, ax = plt.subplots(1, 3, figsize=(18, 6))
    ax[0].imshow(get('terrain_m'), cmap='terrain'); ax[0].set_title('terrain')
    im = ax[1].imshow(get('speedup_10m'), cmap='RdBu_r', vmin=0, vmax=2); ax[1].set_title('speed-up 10 m'); plt.colorbar(im, ax=ax[1])
    im = ax[2].imshow(get('w_50m'), cmap='PuOr', vmin=-2, vmax=2); ax[2].set_title('w 50 m (m/s)'); plt.colorbar(im, ax=ax[2])
    plt.show()